# YOLO26 + EdgeViT

本 Notebook 在 Google Colab 中完成以下工作链：

1. 拉取 Ultralytics YOLO26 和官方 EdgeViT 源码；
2. 检查 YOLO26 基线和 GPU；
3. 兼容新版 timm 的导入接口；
4. 创建 EdgeViT 多尺度骨干适配器；
5. 将 EdgeViT 注册到 Ultralytics；
6. 创建 YOLO26 + EdgeViT YAML；
7. 重启运行时后验证模型构建、前向、反向和 1 epoch 冒烟训练。

第一版使用 EdgeViT_XXS，输出 P3/P4/P5 三层特征。官方 EdgeViT 源码默认要求固定输入尺寸，因此适配器内部统一使用 640×640，再把特征图缩放回 YOLO 当前输入对应的尺寸。


In [1]:
# Colab 新运行时初始化：拉取 YOLO26 和官方 EdgeViT
from pathlib import Path
import subprocess

CONTENT = Path("/content")
ULTRALYTICS = CONTENT / "ultralytics"
EDGEVIT = CONTENT / "edgevit"

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, cwd=cwd, shell=True, check=True)

if not (ULTRALYTICS / ".git").exists():
    run("git clone --depth 1 https://github.com/ultralytics/ultralytics.git /content/ultralytics")
else:
    print("已有 /content/ultralytics，跳过 clone")

if not (EDGEVIT / ".git").exists():
    run("git clone --depth 1 https://github.com/saic-fi/edgevit.git /content/edgevit")
else:
    print("已有 /content/edgevit，跳过 clone")

run("git fetch --depth 1 origin main", cwd=ULTRALYTICS)
run("git checkout --detach origin/main", cwd=ULTRALYTICS)

assert (ULTRALYTICS / "ultralytics" / "cfg" / "models" / "26" / "yolo26.yaml").exists()
assert (EDGEVIT / "src" / "edgevit.py").exists()

run("python -m pip install -q -e /content/ultralytics --no-deps")

print("YOLO26 commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ULTRALYTICS, text=True
).strip())
print("EdgeViT commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=EDGEVIT, text=True
).strip())


In [2]:
# 建立 YOLO26 基线，确认 Colab、PyTorch、CUDA 和模型加载正常
%cd /content/ultralytics

import sys
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

repo = "/content/ultralytics"
if repo not in sys.path:
    sys.path.insert(0, repo)

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Ultralytics:", ultralytics.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

baseline = YOLO("yolo26n.pt")
print("YOLO26 baseline loaded")


In [3]:
# 安装 timm，并记录当前版本。
# EdgeViT 不依赖 timm 中的 EdgeViT 注册名，但官方源码仍使用 timm 的基础层。
%pip install -q timm

import timm

print("timm:", timm.__version__)
print("timm EdgeViT entries:", timm.list_models("*edgevit*"))


In [4]:
# 兼容官方 EdgeViT 源码与新版 timm 的导入路径
from pathlib import Path
import shutil

edgevit_source = Path("/content/edgevit/src/edgevit.py")
source_text = edgevit_source.read_text(encoding="utf-8")

old_imports = '''from timm.models.vision_transformer import _cfg
from timm.models.registry import register_model
from timm.models.layers import trunc_normal_, DropPath, to_2tuple
'''

new_imports = '''from timm.models.vision_transformer import _cfg

try:
    from timm.models.registry import register_model
except ImportError:
    from timm.models._registry import register_model

try:
    from timm.models.layers import trunc_normal_, DropPath, to_2tuple
except ImportError:
    from timm.layers import trunc_normal_, DropPath, to_2tuple
'''

backup = Path(str(edgevit_source) + ".orig")
if not backup.exists():
    shutil.copy2(edgevit_source, backup)

if old_imports in source_text:
    source_text = source_text.replace(old_imports, new_imports, 1)
elif "from timm.models._registry import register_model" in source_text:
    print("EdgeViT 导入兼容补丁已经存在")
else:
    raise RuntimeError("没有找到预期的 EdgeViT timm 导入块，停止以避免误改源码")

edgevit_source.write_text(source_text, encoding="utf-8")
print("patched:", edgevit_source)


In [5]:
# 创建 EdgeViT -> YOLO26 的多尺度骨干适配器
from pathlib import Path

root = Path("/content/ultralytics")
extra = root / "ultralytics" / "nn" / "modules" / "extramodules"
extra.mkdir(parents=True, exist_ok=True)
(extra / "__init__.py").touch()

adapter = r'''# Ultralytics AGPL-3.0 License - https://ultralytics.com/license
# EdgeViT adapter for YOLO26.
from __future__ import annotations

import sys
from functools import partial

import torch
from torch import nn
import torch.nn.functional as F

EDGEVIT_SRC = "/content/edgevit/src"
if EDGEVIT_SRC not in sys.path:
    sys.path.insert(0, EDGEVIT_SRC)

from edgevit import EdgeVit


_VARIANTS = {
    "xxs": {
        "depth": [1, 1, 3, 2],
        "embed_dim": [36, 72, 144, 288],
        "head_dim": 36,
        "sr_ratios": [4, 2, 2, 1],
    },
    "xs": {
        "depth": [1, 1, 3, 1],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
    "s": {
        "depth": [1, 2, 5, 3],
        "embed_dim": [48, 96, 240, 384],
        "head_dim": 48,
        "sr_ratios": [4, 2, 2, 1],
    },
}


class _EdgeViTBackbone(nn.Module):
    # 输出 EdgeViT 的 stride 8/16/32 特征 [P3, P4, P5]。

    def __init__(self, variant="xxs", img_size=640):
        super().__init__()

        cfg = _VARIANTS[variant]
        self.img_size = img_size
        self.channel = list(cfg["embed_dim"][1:])

        self.model = EdgeVit(
            img_size=img_size,
            num_classes=0,
            mlp_ratio=[4] * 4,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-6),
            **cfg,
        )

    def forward(self, x: torch.Tensor):
        original_h, original_w = x.shape[-2:]

        # 官方 PatchEmbed 会检查输入尺寸。YOLO 构建 stride 时可能传入较小张量，
        # 所以统一到 640×640，完成骨干计算后再缩放回当前输入的 P3/P4/P5 尺寸。
        if (original_h, original_w) != (self.img_size, self.img_size):
            x = F.interpolate(
                x,
                size=(self.img_size, self.img_size),
                mode="bilinear",
                align_corners=False,
            )

        # Stage 1: stride 4
        x = self.model.patch_embed1(x)
        x = self.model.pos_drop(x)
        for block in self.model.blocks1:
            x = block(x)

        # Stage 2: stride 8，作为 P3
        x = self.model.patch_embed2(x)
        for block in self.model.blocks2:
            x = block(x)
        p3 = x

        # Stage 3: stride 16，作为 P4
        x = self.model.patch_embed3(x)
        for block in self.model.blocks3:
            x = block(x)
        p4 = x

        # Stage 4: stride 32，作为 P5
        x = self.model.patch_embed4(x)
        for block in self.model.blocks4:
            x = block(x)
        p5 = self.model.norm(x)

        # 让输出尺寸始终与 YOLO 当前输入对应。
        p3 = F.interpolate(
            p3,
            size=(max(original_h // 8, 1), max(original_w // 8, 1)),
            mode="nearest",
        )
        p4 = F.interpolate(
            p4,
            size=(max(original_h // 16, 1), max(original_w // 16, 1)),
            mode="nearest",
        )
        p5 = F.interpolate(
            p5,
            size=(max(original_h // 32, 1), max(original_w // 32, 1)),
            mode="nearest",
        )

        return [p3, p4, p5]


class EdgeViT_XXS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xxs")


class EdgeViT_XS(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("xs")


class EdgeViT_S(_EdgeViTBackbone):
    def __init__(self):
        super().__init__("s")
'''

adapter_path = extra / "edgevit.py"
adapter_path.write_text(adapter, encoding="utf-8")
print("adapter:", adapter_path)


In [6]:
# 独立验证 EdgeViT 适配器的三个输出尺度
import importlib.util
import torch

adapter_path = "/content/ultralytics/ultralytics/nn/modules/extramodules/edgevit.py"
spec = importlib.util.spec_from_file_location("edgevit_adapter", adapter_path)
edgevit_adapter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(edgevit_adapter)

assert issubclass(edgevit_adapter.EdgeViT_XXS, torch.nn.Module)

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = edgevit_adapter.EdgeViT_XXS().to(device).eval()
x = torch.randn(1, 3, 640, 640, device=device)

with torch.inference_mode():
    outputs = backbone(x)

print("channels:", backbone.channel)
for index, output in enumerate(outputs):
    print(index, tuple(output.shape))

assert backbone.channel == [72, 144, 288]
assert [tuple(output.shape[-2:]) for output in outputs] == [
    (80, 80),
    (40, 40),
    (20, 20),
]
print("EdgeViT adapter test passed")


## 修改 Ultralytics 模块注册和 parse_model

下面的单元会：

- 在 ultralytics.nn.modules 导入 EdgeViT 类；
- 在 tasks.py 中将 EdgeViT 标记为特殊骨干；
- 让解析器读取实例化后的 m_.channel；
- 为原文件创建 .bak 备份；
- 创建 EdgeViT YAML。

完成后必须重启 Colab 运行时，清除已经缓存的 Python 模块。


In [7]:
# 注册 EdgeViT 模块，并对 parse_model 做最小修改
from pathlib import Path
import shutil

root = Path("/content/ultralytics")
modules_init = root / "ultralytics" / "nn" / "modules" / "__init__.py"
tasks_path = root / "ultralytics" / "nn" / "tasks.py"

registration = '''
from .extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
'''

modules_text = modules_init.read_text(encoding="utf-8")
if "from .extramodules.edgevit import" not in modules_text:
    modules_backup = Path(str(modules_init) + ".bak")
    if not modules_backup.exists():
        shutil.copy2(modules_init, modules_backup)
    modules_init.write_text(modules_text + registration, encoding="utf-8")
    print("已注册 EdgeViT 到 modules/__init__.py")
else:
    print("modules/__init__.py 已经注册 EdgeViT")

tasks_text = tasks_path.read_text(encoding="utf-8")

if "from ultralytics.nn.modules.extramodules.edgevit import" not in tasks_text:
    import_marker = "from ultralytics.nn.autobackend import check_class_names"
    imports = '''from ultralytics.nn.modules.extramodules.edgevit import (
    EdgeViT_XXS,
    EdgeViT_XS,
    EdgeViT_S,
)
'''
    if import_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到预期导入标记")
    tasks_text = tasks_text.replace(import_marker, imports + "\n" + import_marker, 1)

if "edgevit_modules = frozenset" not in tasks_text:
    loop_marker = '    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):  # from, number, module, args'
    collection = '''    edgevit_modules = frozenset({
        EdgeViT_XXS,
        EdgeViT_XS,
        EdgeViT_S,
    })
'''
    if loop_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 parse_model 循环")
    tasks_text = tasks_text.replace(loop_marker, collection + loop_marker, 1)

    n_marker = "        n = n_ = max(round(n * depth), 1) if n > 1 else n  # depth gain"
    if n_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 depth 计算行")
    tasks_text = tasks_text.replace(n_marker, n_marker + "\n        m_ = None", 1)

    branch_marker = "        if m in base_modules:"
    branch = '''        if m in edgevit_modules:
            if n != 1:
                raise ValueError("EdgeViT backbone only supports repeats=1")
            m_ = m(*args)
            c2 = m_.channel
        elif m in base_modules:
'''
    if branch_marker not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到 base_modules 分支")
    tasks_text = tasks_text.replace(branch_marker, branch, 1)

    instantiate = "        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    replacement = "        if m_ is None:\n            m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    if instantiate not in tasks_text:
        raise RuntimeError("tasks.py 中没有找到模块实例化行")
    tasks_text = tasks_text.replace(instantiate, replacement, 1)

    tasks_backup = Path(str(tasks_path) + ".bak")
    if not tasks_backup.exists():
        shutil.copy2(tasks_path, tasks_backup)

    tasks_path.write_text(tasks_text, encoding="utf-8")
    print("已修改 tasks.py")
else:
    print("tasks.py 已经包含 EdgeViT parse_model 分支")

print("模块注册完成。现在请执行：运行时 -> 重新启动运行时")


In [8]:
# 创建第一版 YOLO26 + EdgeViT 配置
from pathlib import Path

cfg_path = Path(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"
)

cfg_path.write_text(
'''# YOLO26 with EdgeViT-XXS backbone.
# EdgeViT-XXS returns [P3, P4, P5] with channels [72, 144, 288].

nc: 80
end2end: True
reg_max: 1

backbone:
  - [-1, 1, EdgeViT_XXS, []]

head:
  - [0, 1, Index, [72, 0]]
  - [0, 1, Index, [144, 1]]
  - [0, 1, Index, [288, 2]]
  - [[1, 2, 3], 1, Detect, [nc]]
''',
    encoding="utf-8",
)

print("YAML:", cfg_path)


## 重启运行时

执行上一个单元后，请在 Colab 菜单中选择：

运行时 → 重新启动运行时

重启后，从初始化单元重新执行到这里。这样可以清除旧的 ultralytics 和 tasks.py 导入缓存。


In [9]:
# 重启运行时后，构建 YOLO26 + EdgeViT 模型
%cd /content/ultralytics

from ultralytics import YOLO

model = YOLO(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml"
)

model.info(verbose=True)
print("model build passed")


In [10]:
# 前向传播验证
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.model = model.model.to(device).eval()

dummy = torch.randn(1, 3, 640, 640, device=device)

with torch.inference_mode():
    raw_outputs = model.model(dummy)

print("forward passed")


In [ ]:
# 反向传播测试
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = model.model.to(device)
net.train()
net.zero_grad(set_to_none=True)

x = torch.rand(1, 3, 640, 640, device=device)
outputs = net(x)

def collect_tensors(value):
    if isinstance(value, torch.Tensor):
        return [value]

    if isinstance(value, dict):
        tensors = []
        for item in value.values():
            tensors.extend(collect_tensors(item))
        return tensors

    if isinstance(value, (list, tuple)):
        tensors = []
        for item in value:
            tensors.extend(collect_tensors(item))
        return tensors

    return []

tensors = [
    tensor for tensor in collect_tensors(outputs)
    if tensor.requires_grad
]

print("output type:", type(outputs))
if isinstance(outputs, dict):
    print("output keys:", list(outputs.keys()))

print("output tensors:", len(tensors))
print("requires_grad:", [tensor.requires_grad for tensor in tensors])

assert tensors, "没有找到可反向传播的 Tensor"

loss = torch.stack([
    tensor.float().mean()
    for tensor in tensors
]).sum()

print("loss:", loss.item())
loss.backward()

print("backward passed")

### 可选：加载对应骨干的 ImageNet 预训练权重

模型使用 **EdgeViT-XXS**。`True` 时，请先上传可信的、与官方 PyTorch XXS 架构匹配的 ImageNet 权重，并填写路径；权重可以是纯 state dict 或包含 `model`/`state_dict` 的 checkpoint。未找到可核实的官方 XXS 下载地址，因此此处不自动下载，也不使用 XS/S 或第三方任务权重代替。

默认 `False`，不下载或读取骨干权重。检测头始终随机初始化，两个模式都不恢复旧训练进度。切换时重跑下面的设置格和冒烟格；只跑一轮 COCO8 来验证流程，不用于比较精度。


In [ ]:
# @title 冒烟前：选择 EdgeViT-XXS 初始化方式
USE_IMAGENET_PRETRAINED = False  # @param {type:"boolean"}
EDGEVIT_XXS_CHECKPOINT = '/content/edgevit_xxs_imagenet.pth'  # @param {type:"string"}


def load_vit_backbone(detector):
    from argparse import Namespace

    checkpoint_path = Path(EDGEVIT_XXS_CHECKPOINT).expanduser()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f'找不到 {checkpoint_path}。请上传与官方 PyTorch EdgeViT-XXS 架构匹配的 '
            'ImageNet checkpoint 并填写路径，或将预训练开关设为 False。'
        )
    # 只使用可信的 checkpoint；允许 PyTorch 读取常见的训练参数 Namespace。
    with torch.serialization.safe_globals([Namespace]):
        payload = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if not isinstance(payload, dict):
        raise TypeError('需要 PyTorch state_dict 或包含 model/state_dict 的 checkpoint。')
    state = payload.get('model', payload.get('state_dict', payload))
    if not isinstance(state, dict):
        raise TypeError('checkpoint 中的 model/state_dict 不是参数映射。')

    # 处理 DDP 的 module. 前缀，只排除分类/蒸馏头，不跳过任何骨干层。
    state = {key.removeprefix('module.'): value for key, value in state.items()}
    state = {
        key: value for key, value in state.items()
        if not key.startswith(('head.', 'head_dist.'))
    }
    detector.model.model[0].model.load_state_dict(state, strict=True)
    print(f'EdgeViT-XXS 骨干已加载：{len(state)} 项参数和缓冲区')


In [ ]:
# 1 epoch 冒烟训练：每次从 YAML 重新建模，不复用前面代理梯度测试过的模型。
from pathlib import Path
from tempfile import TemporaryDirectory
import torch
from ultralytics import YOLO

use_imagenet = USE_IMAGENET_PRETRAINED
smoke_model = YOLO('/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml')
run_name = 'yolo26_edgevit_imagenet_smoke' if use_imagenet else 'yolo26_edgevit_smoke'
print('初始化:', 'ImageNet 骨干 + 随机检测头' if use_imagenet else '全模型随机初始化')

with TemporaryDirectory(prefix='edgevit-init-') as temp_dir:
    if use_imagenet:
        load_vit_backbone(smoke_model)
        # Ultralytics 从 YAML 训练时会重建模型；借助自己生成的本地 .pt 传入初始化权重。
        # 临时文件保留到训练结束，之后自动清理，不影响 runs/ 中的 best.pt。
        init_path = Path(temp_dir) / 'yolo26_edgevit_imagenet_init.pt'
        smoke_model.save(str(init_path))
        smoke_model = YOLO(str(init_path), task='detect')

    smoke_model.train(
        data='coco8.yaml', epochs=1, imgsz=640, batch=2, workers=2,
        device=0 if torch.cuda.is_available() else 'cpu',
        pretrained=use_imagenet, resume=False,
        project='/content/runs', name=run_name,
    )


In [ ]:
# 检查训练结果和工作区改动
from pathlib import Path
import subprocess

best = Path(smoke_model.trainer.best)
print("best checkpoint:", best, "exists:", best.exists())

print("\nUltralytics working tree:")
subprocess.run(["git", "status", "--short"], cwd="/content/ultralytics", check=False)

print("\nDiff summary:")
subprocess.run(["git", "diff", "--stat"], cwd="/content/ultralytics", check=False)


## D 段：方案 2 — EdgeViT-XXS + 轻量 FPN/PAN + Detect

先完成原 Notebook 的初始化、适配器和模型注册，按原说明重启 Python 会话，再从 D1 开始。**不需要先跑原冒烟训练。** D1 单独生成 `yolo26_edgevit_light_neck.yaml`，原 YAML 和历史输出不变。

Neck 中间宽度为 **96**：1×1 横向投影 → 两次 nearest 上采样（FPN）→ 两次 stride 2 深度卷积下采样（PAN）。每次融合使用 `Concat(dim=1)` → `DWConv(3×3, groups=192)` → `Conv(1×1)`。末端将三个尺度投影回 **72/144/288** 通道，使 Detect 的输入、内部参数形状及端到端检测逻辑与原方案一致。不新增自定义层或 `parse_model()` 补丁。

沿用原适配器的三个阶段输出和内部 640×640 计算，不改取层逻辑。 Neck 要求输入 H/W 为 32 的倍数以对齐拼接；常规 YOLO 数据预处理负责对齐。


In [ ]:
# D1. 保留方案 1，单独生成轻量 Neck YAML。
from pathlib import Path

CFG_PATH = Path('/content/ultralytics/ultralytics/cfg/models/26/yolo26_edgevit.yaml')

LIGHT_NECK_YAML = '''# YOLO26 + EdgeViT-XXS + lightweight FPN/PAN.
nc: 80
end2end: true
reg_max: 1

backbone:
  - [-1, 1, EdgeViT_XXS, []]       # 0: same backbone as the no-neck model

head:
  - [0, 1, Index, [72, 0]]         # 1: F3 (stride 8)
  - [0, 1, Index, [144, 1]]        # 2: F4 (stride 16)
  - [0, 1, Index, [288, 2]]        # 3: F5 (stride 32)
  - [1, 1, Conv, [96, 1, 1]]      # 4: L3
  - [2, 1, Conv, [96, 1, 1]]      # 5: L4
  - [3, 1, Conv, [96, 1, 1]]      # 6: L5

  - [6, 1, nn.Upsample, [null, 2, nearest]]  # 7: L5 -> stride 16
  - [[7, 5], 1, Concat, [1]]       # 8: 96 + 96 = 192
  - [-1, 1, DWConv, [192, 3, 1]]   # 9: groups=192
  - [-1, 1, Conv, [96, 1, 1]]     # 10: T4
  - [10, 1, nn.Upsample, [null, 2, nearest]]  # 11: T4 -> stride 8
  - [[11, 4], 1, Concat, [1]]      # 12: 192 channels
  - [-1, 1, DWConv, [192, 3, 1]]   # 13
  - [-1, 1, Conv, [96, 1, 1]]     # 14: T3

  - [14, 1, DWConv, [96, 3, 2]]   # 15: T3 -> stride 16, groups=96
  - [[15, 10], 1, Concat, [1]]     # 16: 192 channels
  - [-1, 1, DWConv, [192, 3, 1]]   # 17
  - [-1, 1, Conv, [96, 1, 1]]     # 18: fused stride 16
  - [18, 1, DWConv, [96, 3, 2]]   # 19: fused stride 16 -> stride 32
  - [[19, 6], 1, Concat, [1]]      # 20: 192 channels
  - [-1, 1, DWConv, [192, 3, 1]]   # 21
  - [-1, 1, Conv, [96, 1, 1]]     # 22: fused stride 32

  - [14, 1, Conv, [72, 1, 1]]      # 23: P3 (match original Detect input)
  - [18, 1, Conv, [144, 1, 1]]     # 24: P4
  - [22, 1, Conv, [288, 1, 1]]     # 25: P5
  - [[23, 24, 25], 1, Detect, [nc]]  # 26: same Detect as the no-neck model
'''

if not CFG_PATH.is_file():
    raise FileNotFoundError('找不到方案 1 YAML，请先完成原 Notebook 的 YAML 生成和注册步骤。')
LIGHT_NECK_CFG = CFG_PATH.with_name('yolo26_edgevit_light_neck.yaml')
if LIGHT_NECK_CFG.exists() and LIGHT_NECK_CFG.read_text(encoding='utf-8') != LIGHT_NECK_YAML:
    raise RuntimeError(f'不覆盖不同内容的已有配置：{LIGHT_NECK_CFG}')
LIGHT_NECK_CFG.write_text(LIGHT_NECK_YAML, encoding='utf-8')
print('方案 1:', CFG_PATH)
print('方案 2:', LIGHT_NECK_CFG)


In [ ]:
# D2. 构建两个模型，验证解析、分支连接、通道、分辨率、stride 和真实前向。
import torch
from ultralytics import YOLO
from ultralytics.nn.modules import Concat, Conv, DWConv, Detect


def check_light_neck(cfg_no_neck, cfg_light_neck, device):
    plain = YOLO(str(cfg_no_neck), task='detect').model.to(device).eval()
    light = YOLO(str(cfg_light_neck), task='detect').model.to(device).eval()
    base_head, light_head = plain.model[-1], light.model[-1]
    assert type(plain.model[0]) is type(light.model[0])
    assert type(light.model[0]).__name__ == 'EdgeViT_XXS'
    assert plain.model[0].channel == light.model[0].channel == [72, 144, 288]
    assert plain.yaml['backbone'] == light.yaml['backbone']
    assert plain.yaml['head'][:3] == light.yaml['head'][:3]  # same Index taps
    assert type(base_head) is type(light_head) is Detect
    assert (base_head.nc, base_head.reg_max, base_head.end2end, base_head.legacy) == (
        light_head.nc, light_head.reg_max, light_head.end2end, light_head.legacy,
    ) == (80, 1, True, True)
    assert {k: v.shape for k, v in base_head.state_dict().items()} == {
        k: v.shape for k, v in light_head.state_dict().items()
    }
    assert plain.stride.tolist() == light.stride.tolist() == [8.0, 16.0, 32.0]
    assert base_head.f == [1, 2, 3] and light_head.f == [23, 24, 25]
    assert all(isinstance(light.model[i], Conv) and light.model[i].conv.kernel_size == (1, 1)
               for i in (4, 5, 6, 10, 14, 18, 22, 23, 24, 25))
    for i, channels, step in [(9,192,1), (13,192,1), (15,96,2), (17,192,1), (19,96,2), (21,192,1)]:
        layer = light.model[i]
        assert isinstance(layer, DWConv)
        conv = layer.conv
        assert conv.groups == conv.in_channels == conv.out_channels == channels
        assert conv.kernel_size == (3, 3) and conv.stride == (step, step)
    assert all(isinstance(light.model[i], Concat) and light.model[i].d == 1 for i in (8, 12, 16, 20))
    assert all(isinstance(light.model[i], torch.nn.Upsample) and light.model[i].mode == 'nearest'
               and light.model[i].scale_factor == 2 for i in (7, 11))

    # 每个条目记录 (通道数, stride)，包括所有 Concat 的输出和两次上下采样。
    expected = {
        1:(72,8), 2:(144,16), 3:(288,32), 4:(96,8), 5:(96,16), 6:(96,32),
        7:(96,16), 8:(192,16), 9:(192,16), 10:(96,16),
        11:(96,8), 12:(192,8), 13:(192,8), 14:(96,8),
        15:(96,16), 16:(192,16), 17:(192,16), 18:(96,16),
        19:(96,32), 20:(192,32), 21:(192,32), 22:(96,32),
        23:(72,8), 24:(144,16), 25:(288,32),
    }
    shapes = {}
    def remember(i):
        return lambda layer, args, out: shapes.update({i: tuple(out.shape)})
    hooks = [light.model[i].register_forward_hook(remember(i)) for i in expected]
    try:
        with torch.inference_mode():
            for h, w in [(640, 640), (384, 640)]:
                shapes.clear()
                x = torch.rand(1, 3, h, w, device=device)
                num_pos = sum((h // s) * (w // s) for s in (8, 16, 32))
                for label, net in [('no neck', plain), ('light neck', light)]:
                    det, raw = net(x)
                    assert det.shape == (1, 300, 6) and torch.isfinite(det).all().item()
                    assert set(raw) == {'one2many', 'one2one'}
                    for branch in raw.values():
                        assert branch['boxes'].shape == (1, 4, num_pos)
                        assert branch['scores'].shape == (1, 80, num_pos)
                        actual = [tuple(f.shape) for f in branch['feats']]
                        assert actual == [(1,c,h//s,w//s) for c,s in ((72, 8),(144, 16),(288, 32))]
                    print(f'{label} {h}x{w}: Detect inputs={actual}; positions={num_pos}')
                for i, (c, s) in expected.items():
                    assert shapes[i] == (1, c, h//s, w//s), (i, shapes[i], c, s)
    finally:
        for hook in hooks:
            hook.remove()
    print('解析、FPN/PAN 全部中间层、Detect 输入及前向检查均通过。')


DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
check_light_neck(CFG_PATH, LIGHT_NECK_CFG, DEVICE)


In [ ]:
# D3（可选）. 同一设备、同一输入、FP32、batch=1，比较参数、GFLOPs、模型前向延迟。
# eval（不 fuse），含 Detect 后处理；不含读图、预处理、传输。GFLOPs 为估算。
from statistics import median
from time import perf_counter
from ultralytics.utils.torch_utils import get_flops

PROFILE_IMGSZ = 640
x_profile = torch.zeros(1, 3, PROFILE_IMGSZ, PROFILE_IMGSZ, device=DEVICE)


def sync_device():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)


rows = []
for label, cfg in [('no neck', CFG_PATH), ('light neck', LIGHT_NECK_CFG)]:
    net_profile = YOLO(str(cfg), task='detect').model.to(DEVICE).float().eval()
    params = sum(p.numel() for p in net_profile.parameters())
    gflops = get_flops(net_profile, imgsz=PROFILE_IMGSZ)
    seconds = []
    with torch.inference_mode():
        for _ in range(10):
            net_profile(x_profile)
        sync_device()
        for _ in range(30):
            start = perf_counter()
            net_profile(x_profile)
            sync_device()
            seconds.append(perf_counter() - start)
    rows.append((label, params, gflops, 1e3 * median(seconds)))
    del net_profile

del x_profile
for label, p, g, ms in rows:
    print(f'{label:10s} | params={p:,} | GFLOPs={f"{g:.2f}" if g else "N/A"} | median={ms:.2f} ms')
print(f'Neck 参数增量：{rows[1][1] - rows[0][1]:,}')


## E 段（可选）：同一骨干下的两种方案对照训练

复制同一组 Backbone 和 Detect 初始权重，只有轻量 Neck 独立初始化；使用同一个数据集、分辨率、seed 和训练参数。默认不用预训练。启用预训练时，先运行原 Notebook 的预训练设置格并填写可信的 `EDGEVIT_XXS_CHECKPOINT` 路径。 无论是否预训练，两边都从第 1 个 epoch 开始，不续训。

默认使用 COCO8 跑一轮，仅验证链路。正式对比时请把 `COMPARE_DATA` 和训练轮数改为实验设置。参数量和前向延迟可用 D3 比较；原方案的冒烟记录不会被覆盖。


In [ ]:
# E1. 运行这一格会训练两次，结果分别保存；不影响 C 段原有的冒烟实验。
from tempfile import TemporaryDirectory

COMPARE_DATA = 'coco8.yaml'
COMPARE_EPOCHS = 1
COMPARE_IMGSZ = 640
COMPARE_BATCH = 2
COMPARE_USE_IMAGENET = False  # @param {type:"boolean"}

compare_train_args = dict(
    data=COMPARE_DATA, epochs=COMPARE_EPOCHS, imgsz=COMPARE_IMGSZ, batch=COMPARE_BATCH,
    workers=2, device=0 if torch.cuda.is_available() else 'cpu', plots=False,
    seed=0, deterministic=True, resume=False, pretrained=COMPARE_USE_IMAGENET,
    project='/content/runs/edgevit_neck_compare',
)

if COMPARE_USE_IMAGENET and not callable(globals().get('load_vit_backbone')):
    raise RuntimeError('请先运行原 Notebook 的预训练设置格，定义 load_vit_backbone。')

plain_compare = YOLO(str(CFG_PATH), task='detect')
light_compare = YOLO(str(LIGHT_NECK_CFG), task='detect')
if COMPARE_USE_IMAGENET:
    load_vit_backbone(plain_compare)  # 复用现有模型专用 loader；只下载或读取一次。
# 复制所有骨干和 Detect 参数/缓冲区，消除建模顺序造成的随机初始化差异。
for i in (0, -1):
    light_compare.model.model[i].load_state_dict(plain_compare.model.model[i].state_dict(), strict=True)

comparison = {}
with TemporaryDirectory(prefix='edgevit-neck-compare-') as tmp:
    recipes = [('no_neck', plain_compare), ('light_neck', light_compare)]
    for label, detector in recipes:
        # 使用 .pt 入口，确保 YOLO26 trainer 重建模型时保留上面复制的初始化。
        init = Path(tmp) / f'{label}.pt'
        detector.save(str(init))
        run = YOLO(str(init), task='detect')
        result = run.train(**compare_train_args, name=label + ('_imagenet' if COMPARE_USE_IMAGENET else '_random'))
        comparison[label] = (Path(run.trainer.best), dict(result.results_dict))
        assert comparison[label][0].is_file()
        del run

for label, (best_path, metrics) in comparison.items():
    print(label, best_path)
    for metric in ('metrics/mAP50(B)', 'metrics/mAP50-95(B)'):
        print(f'  {metric}: {metrics.get(metric, "N/A")}')
